In [ ]:
import numpy as np
import torch

# 1. 텍스트 분류(Text Classification) 개요

이번 챕터에서는 파이토치로 인공 신경망을 이용한 **텍스트 분류(Text Classification)**를 실습한다. 본격적인 구현에 앞서, 딥 러닝으로 텍스트 분류를 수행할 때 어떤 작업과 구성으로 진행되는지 먼저 정리한다.

## 훈련 데이터에 대한 이해

텍스트 분류는 **지도 학습(Supervised Learning)**에 속한다. 지도 학습의 훈련 데이터는 **레이블(label)**이라는 이름으로 정답이 미리 적혀 있는 데이터로 구성된다. 기계는 정답이 적힌 문제지를 열심히 공부하고, 향후 정답이 없는 문제에 대해서도 정답을 예측해서 대답하게 되는 메커니즘이다.

예를 들어 스팸 메일 분류기의 훈련 데이터는 메일의 내용(텍스트)과 해당 메일이 정상/스팸인지가 적힌 레이블, 두 개의 열로 이루어진다.

| 텍스트 (메일의 내용) | 레이블 (스팸 여부) |
|:---|:---:|
| 당신에게 드리는 마지막 혜택! ... | 스팸 메일 |
| 내일 뵐 수 있을지 확인 부탁... | 정상 메일 |
| 쉿! 혼자 보세요... | 스팸 메일 |
| 언제까지 답장 가능할... | 정상 메일 |
| (광고) 멋있어질 수 있는... | 스팸 메일 |

이런 메일 샘플이 20,000개 있다고 하자. 데이터가 깔끔하고 모델이 잘 설계되었다면, 학습이 끝난 모델은 훈련 데이터에 없던 새로운 메일 텍스트가 주어졌을 때도 레이블을 예측할 수 있게 된다.

## 훈련 데이터와 테스트 데이터

전체 데이터를 전부 훈련에 쓰기보다는, 일부를 **테스트용으로 남겨 두는 것**이 바람직하다. 20,000개 중 18,000개로 훈련하고 2,000개는 보류해 두었다가, 훈련이 끝난 뒤 레이블을 가린 채 모델에게 맞혀 보라고 요구하고 정답률을 계산하는 식이다.

이후 예제에서는 분류하고자 하는 텍스트 열을 $X$, 레이블 열을 $y$ 라 명명하고, 이를 훈련 데이터(`X_train`, `y_train`)와 테스트 데이터(`X_test`, `y_test`)로 분리한다. 모델은 `X_train` 과 `y_train` 을 학습하고, `X_test` 에 대해 레이블을 예측한 뒤 `y_test` 와 비교해 정답률을 계산한다. 여기에 더해 이번 실습에서는 학습 도중의 성능 점검을 위한 **검증 데이터**(`X_valid`, `y_valid`)까지 세 갈래로 분리한다.

## 단어에 대한 인덱스 부여

파이토치의 `nn.Embedding()` 은 **단어 각각에 정수가 맵핑된 입력**에 대해 임베딩을 수행한다. 따라서 텍스트를 모델에 넣기 전에 각 단어에 고유한 정수 인덱스를 부여해야 한다. 대표적인 방법은 단어를 **등장 빈도수 순으로 정렬하고 순차적으로 인덱스를 부여**하는 것이다.

빈도순 정렬의 장점은 **등장 빈도가 낮은 단어를 쉽게 제거**할 수 있다는 점이다. 예를 들어 25,000개 단어에 빈도 높은 순서로 1부터 25,000까지 인덱스를 부여하면 인덱스가 곧 등수가 되므로, 전처리에서 1,000을 넘는 인덱스의 단어를 잘라내면 상위 1,000개 단어만 남길 수 있다.

## RNN의 다-대-일(Many-to-One) 구조

텍스트 분류는 RNN의 **다-대-일(Many-to-One)** 문제에 속한다. 모든 시점(time step)에서 입력을 받지만, **최종 시점의 RNN 셀만 은닉 상태를 출력**하고 이것이 출력층으로 가서 활성화 함수를 통해 정답을 고른다.

```python
# 실제 RNN 은닉층을 추가하는 코드
nn.RNN(input_size, hidden_size, batch_first=True)
```

텍스트 분류 관점에서 각 인자를 해석하면 다음과 같다. (RNN의 변형인 LSTM, GRU도 동일하다.)

| 인자 | 텍스트 분류에서의 의미 |
|:---:|:---|
| `hidden_size` | 출력의 크기 (output_dim) |
| `timesteps` | 시점의 수 = 각 문서에서의 단어 수 |
| `input_size` | 입력의 크기 = 각 단어의 벡터 표현의 차원 수 |

## 이진 분류와 다중 클래스 분류

두 개의 선택지 중 정답을 고르면 **이진 분류(Binary Classification)**, 세 개 이상이면 **다중 클래스 분류(Multi-Class Classification)**다. 이진 분류는 출력층의 활성화 함수로 시그모이드를, 다중 클래스는 소프트맥스를 사용하며, 클래스가 $N$ 개라면 출력층 뉴런의 수를 $N$ 개로 한다.

> **여기서 잠깐!** 소프트맥스 함수로도 이진 분류를 할 수 있다. **출력층에 뉴런을 2개** 배치하면 된다. 이번 챕터의 모든 실습(긍정/부정 감성 분류)이 바로 이 방식을 사용한다. 손실 함수는 `nn.CrossEntropyLoss()` 를 쓰는데, 이 함수는 **소프트맥스 함수까지 포함**하고 있으므로 모델의 `forward` 는 소프트맥스를 거치지 않은 **로짓(logits)**을 그대로 반환해야 한다.

# 2. 텍스트 분류 파이프라인 : 데이터 전처리

교안 13-02(네이버 영화 리뷰 + LSTM), 13-03(IMDB 리뷰 + GRU), 13-04(IMDB 리뷰 + 1D CNN)는 모델만 다를 뿐 **전처리 파이프라인이 완전히 동일**하다. 이 흐름을 한 번 확실히 익혀 두면 모델을 갈아 끼우는 것은 어렵지 않다.

```text
데이터 로드 → 정제 → 분할(train/valid/test) → 토큰화
→ 단어 집합(Vocab) → 정수 인코딩 → 패딩 → 텐서 변환/데이터로더
```

> **실습 데이터에 대한 노트** : 교안은 IMDB 리뷰 5만 개(13-03·13-04) 또는 네이버 영화 리뷰 20만 개(13-02)를 인터넷에서 내려받아 사용한다. 이 학습노트에서는 어떤 환경에서도 즉시 실행되도록 **소형 영어 리뷰 코퍼스 100문장을 코드 안에 직접 정의**해 동일한 파이프라인을 재현한다. 실제 IMDB 데이터로 바꾸려면 아래 주석 코드를 사용하면 된다.
>
> ```python
> # import urllib.request, pandas as pd
> # urllib.request.urlretrieve(
> #     "https://raw.githubusercontent.com/ukairia777/pytorch-nlp-tutorial/main/"
> #     "10.%20RNN%20Text%20Classification/dataset/IMDB%20Dataset.csv",
> #     filename="IMDB Dataset.csv")
> # df = pd.read_csv('IMDB Dataset.csv')   # review / sentiment 2개 열, 총 5만 개
> ```

## 데이터 로드

긍정 리뷰 50개, 부정 리뷰 50개를 정의하고 `review` / `sentiment` 두 개의 열을 가진 데이터프레임으로 만든다. IMDB 데이터셋과 같은 형태다.

In [ ]:
import pandas as pd

pos_reviews = [
    "this movie was a true masterpiece and I loved every minute",
    "an amazing film with great acting and a brilliant story",
    "the story was touching and the acting was simply great",
    "a brilliant masterpiece that I would happily watch again",
    "I loved the characters and the beautiful ending of this film",
    "great direction and amazing performances made this film shine",
    "the best movie I have seen this year truly amazing",
    "a beautiful story with touching moments and great music",
    "wonderful acting and a brilliant script made it a joy to watch",
    "I enjoyed every scene and the ending was truly beautiful",
    "this film is a joy from start to finish highly recommended",
    "amazing visuals and a great story kept me hooked",
    "the performances were wonderful and the story was touching",
    "a great film with brilliant acting and a satisfying ending",
    "truly one of the best films ever made simply wonderful",
    "I loved this movie the acting was amazing and the story great",
    "highly recommended a masterpiece of storytelling and emotion",
    "the music was beautiful and the acting was superb",
    "superb direction wonderful characters and a touching story",
    "an unforgettable film full of joy and brilliant moments",
    "every scene was beautiful and the characters felt real",
    "the script was brilliant and the ending left me smiling",
    "a wonderful movie that the whole family enjoyed together",
    "great pacing amazing acting and a story full of heart",
    "this is cinema at its best a truly great experience",
    "the film was moving and the performances were outstanding",
    "outstanding visuals and a superb story I loved it",
    "a touching and beautiful film that deserves every award",
    "I smiled through the whole movie simply a great time",
    "brilliant from the first scene to the last truly superb",
    "the director made a wonderful film with superb acting",
    "an outstanding movie with a touching story and great music",
    "I enjoyed the brilliant script and the amazing visuals",
    "the best film of the year with wonderful performances",
    "a superb masterpiece that I recommended to all my friends",
    "the acting was outstanding and the story kept me smiling",
    "beautiful music great acting and an unforgettable ending",
    "I loved the joy and heart in every single scene",
    "a moving story with amazing characters and superb pacing",
    "this wonderful movie deserves an award for its brilliant script",
    "the film was amazing and I enjoyed every moment of it",
    "great characters a touching ending and beautiful direction",
    "truly a masterpiece the best movie I watched this year",
    "the performances were superb and the visuals were outstanding",
    "I recommended this film because the story is simply wonderful",
    "an amazing experience with brilliant direction and great heart",
    "the ending was touching and the whole film felt beautiful",
    "superb storytelling wonderful acting and a joy to watch",
    "I loved how the brilliant story kept me hooked until the ending",
    "a great movie full of heart with outstanding performances",
]

neg_reviews = [
    "this movie was a complete waste of time and money",
    "terrible acting and a boring story made it painful to watch",
    "the plot was dumb and the acting was simply awful",
    "a boring film with terrible pacing and a weak story",
    "I hated the characters and the ending was disappointing",
    "poor direction and awful performances ruined this film",
    "the worst movie I have seen this year truly terrible",
    "a dull story with boring moments and annoying music",
    "awful acting and a lazy script made it hard to watch",
    "I regretted every scene and the ending was disappointing",
    "this film is a mess from start to finish not recommended",
    "cheap visuals and a dumb story bored me completely",
    "the performances were awful and the story was pointless",
    "a bad film with terrible acting and a predictable ending",
    "truly one of the worst films ever made simply awful",
    "I hated this movie the acting was terrible and the story dumb",
    "not worth watching a disaster of storytelling and emotion",
    "the music was annoying and the acting was horrible",
    "horrible direction boring characters and a pointless story",
    "a forgettable film full of boredom and lazy moments",
    "every scene was dull and the characters felt fake",
    "the script was lazy and the ending left me angry",
    "a terrible movie that the whole family regretted watching",
    "bad pacing awful acting and a story with no heart",
    "this is cinema at its worst a truly bad experience",
    "the film was boring and the performances were disappointing",
    "cheap effects and a horrible story I hated it",
    "a dull and predictable film that deserves no award",
    "I yawned through the whole movie simply a bad time",
    "terrible from the first scene to the last truly horrible",
    "the director made a horrible film with awful acting",
    "a disappointing movie with a pointless story and annoying music",
    "I regretted the lazy script and the cheap visuals",
    "the worst film of the year with terrible performances",
    "an awful mess that I would never recommend to my friends",
    "the acting was horrible and the story left me angry",
    "annoying music bad acting and a forgettable ending",
    "I hated the boredom and the dumb plot in every scene",
    "a dull story with fake characters and terrible pacing",
    "this disappointing movie deserves no award for its lazy script",
    "the film was awful and I regretted every moment of it",
    "weak characters a predictable ending and poor direction",
    "truly a disaster the worst movie I watched this year",
    "the performances were terrible and the visuals were cheap",
    "not recommended because the story is simply pointless",
    "a painful experience with poor direction and no heart",
    "the ending was predictable and the whole film felt fake",
    "weak storytelling horrible acting and a bore to watch",
    "I hated how the dumb plot dragged on until the ending",
    "a bad movie full of boredom with disappointing performances",
]

df = pd.DataFrame({
    'review': pos_reviews + neg_reviews,
    'sentiment': ['positive'] * len(pos_reviews) + ['negative'] * len(neg_reviews)
})
# 긍정/부정이 앞뒤로 몰려 있지 않도록 행을 섞는다.
df = df.sample(frac=1, random_state=0).reset_index(drop=True)
df

## 데이터 점검 : 결측값과 레이블 분포

데이터에 **결측값**이 있는지 확인한다. 데이터프레임의 정보를 요약해 주는 `info()` 를 쓰거나, `.isnull().values.any()` 로 결측값 존재 여부를 바로 확인할 수 있다. `False` 가 출력되면 결측값이 없다는 의미다.

In [ ]:
df.info()

In [ ]:
print('결측값 여부 :', df.isnull().values.any())

레이블이 균등한지도 확인한다. 클래스 불균형이 심하면 모델이 다수 클래스만 찍어도 정확도가 높게 나오는 착시가 생기므로, 분류 문제에서는 반드시 짚고 넘어가야 하는 항목이다.

In [ ]:
print('레이블 개수')
print(df.groupby('sentiment').size().reset_index(name='count'))

## 레이블 정수 변환

레이블이 문자열 `'positive'` / `'negative'` 로 되어 있으므로 각각 정수 1, 0으로 변환한다.

> **버전 노트** : 교안은 `replace(['positive','negative'], [1, 0])` 만 사용하지만, 최신 pandas에서는 결과 dtype이 `object` 로 남아 이후 `torch.tensor()` 변환에서 오류가 날 수 있다. `.astype(int)` 를 붙여 정수형으로 확정해 주면 버전과 무관하게 안전하다.

In [ ]:
df['sentiment'] = df['sentiment'].replace(['positive', 'negative'], [1, 0]).astype(int)
df.head()

## 훈련 / 검증 / 테스트 데이터 분할

리뷰 열은 `X_data`, 레이블 열은 `y_data` 에 저장한 뒤 데이터를 나눈다. 교안과 동일하게 우선 훈련:테스트를 **5:5**로 나누고, 훈련 데이터를 다시 **8:2**로 나누어 검증 데이터를 만든다.

sklearn의 `train_test_split` 은 데이터를 나눌 때 매우 많이 쓰는 도구이므로 꼭 기억해 두자. 분할 시 **레이블의 비율을 유지**하고 싶다면 레이블 데이터를 `stratify` 인자에 넘기면 된다.

In [ ]:
from sklearn.model_selection import train_test_split

X_data = df['review']
y_data = df['sentiment']
print('영화 리뷰의 개수: {}'.format(len(X_data)))
print('레이블의 개수: {}'.format(len(y_data)))

# 훈련:테스트 = 5:5 → 훈련:검증 = 8:2
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.5, random_state=0, stratify=y_data)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=0, stratify=y_train)

print('훈련 데이터 :', len(X_train), '/ 검증 데이터 :', len(X_valid), '/ 테스트 데이터 :', len(X_test))

In [ ]:
print('--------훈련 데이터의 비율-----------')
print(f'부정 리뷰 = {round(y_train.value_counts()[0]/len(y_train) * 100, 3)}%')
print(f'긍정 리뷰 = {round(y_train.value_counts()[1]/len(y_train) * 100, 3)}%')
print('--------검증 데이터의 비율-----------')
print(f'부정 리뷰 = {round(y_valid.value_counts()[0]/len(y_valid) * 100, 3)}%')
print(f'긍정 리뷰 = {round(y_valid.value_counts()[1]/len(y_valid) * 100, 3)}%')
print('--------테스트 데이터의 비율-----------')
print(f'부정 리뷰 = {round(y_test.value_counts()[0]/len(y_test) * 100, 3)}%')
print(f'긍정 리뷰 = {round(y_test.value_counts()[1]/len(y_test) * 100, 3)}%')

세 데이터 모두 긍정/부정 레이블이 50:50으로 균등하게 유지된 채 분할되었다.

## 한국어 데이터라면? (교안 13-02 노트)

교안 13-02는 **네이버 영화 리뷰(NSMC)** 데이터를 다룬다. 한국어 데이터는 영어와 달리 두 가지 전처리가 추가로 필요하다.

**첫째, 정규 표현식으로 한글만 남기는 정제.** 영어의 알파벳을 나타내는 정규 표현식이 `[a-zA-Z]` 이듯, 한글은 자음 `ㄱ ~ ㅎ`, 모음 `ㅏ ~ ㅣ`, 완성형 음절 `가 ~ 힣` 으로 범위를 지정할 수 있다. 먼저 영어 예시부터 확인한다.

In [ ]:
import re

# 알파벳과 공백을 제외하고 모두 제거
eng_text = "do!!! you expect... people~ to~ read~ the FAQ, etc. and actually accept hard~! atheism?@@"
print(re.sub(r'[^a-zA-Z ]', '', eng_text))

In [ ]:
# 한글과 공백을 제외하고 모두 제거
kor_text = "와!! 이런 것도 영화라고...ㅋㅋ 차라리 뮤직비디오를 만드는 게 나을 뻔 111"
print(re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣 ]', '', kor_text))

띄어쓰기는 유지되면서 구두점·숫자·영문이 제거된다. 정제 후 한글이 하나도 없던 리뷰는 빈(empty) 값이 되므로, 교안에서는 공백만 남은 행을 `np.nan` 으로 바꾼 뒤 `dropna()` 로 제거하는 과정을 함께 수행한다.

**둘째, 형태소 분석기를 이용한 토큰화.** 한국어는 교착어라서 영어처럼 띄어쓰기 기준으로 토큰화하면 조사가 붙은 형태가 전부 다른 단어로 취급된다. 그래서 KoNLPy의 **Mecab** 같은 형태소 분석기로 토큰화하고, 그 과정에서 조사·접속사 등의 **불용어(stopwords)**를 제거한다.

```python
# from konlpy.tag import Mecab            # Colab에서는 별도 설치 과정 필요
# stopwords = ['도', '는', '다', '의', '가', '이', '은', '한', '에', '하', '고',
#              '을', '를', '인', '듯', '과', '와', '네', '들', '지', '임', '게']
# mecab = Mecab()
# mecab.morphs('와 이런 것도 영화라고 차라리 뮤직비디오를 만드는 게 나을 뻔')
# 결과 : ['와', '이런', '것', '도', '영화', '라고', '차라리', '뮤직', '비디오',
#         '를', '만드', '는', '게', '나을', '뻔']
```

> **환경 노트** : KoNLPy(Mecab)는 JVM·시스템 사전 설치가 필요해 이 노트에서는 코드 패턴만 남긴다. 형태소 분석 결과 리스트가 준비된 뒤부터는 아래의 영어 파이프라인과 **완전히 동일한 과정**(단어 집합 → 정수 인코딩 → 패딩)을 밟는다.

## 토큰화

영어 데이터는 단어 토큰화를 수행한다. 교안은 NLTK의 `word_tokenize` 를 사용하는데, 여기서는 외부 리소스 다운로드 없이 동작하도록 정규 표현식 기반의 간단한 토크나이저를 정의한다. 토큰화와 동시에 `.lower()` 로 소문자화도 수행한다.

In [ ]:
# 교안의 방식 (Colab에서 사용 시 주석 해제)
# import nltk
# from nltk.tokenize import word_tokenize
# nltk.download('punkt')

def simple_tokenize(text):
    # 소문자화 후, 영문/숫자/어포스트로피의 연속을 하나의 토큰으로 추출한다.
    return re.findall(r"[a-z0-9']+", text.lower())

def tokenize(sentences):
    tokenized_sentences = []
    for sent in sentences:
        tokenized_sentences.append(simple_tokenize(sent))   # 교안: word_tokenize(sent) + 소문자화
    return tokenized_sentences

tokenized_X_train = tokenize(X_train)
tokenized_X_valid = tokenize(X_valid)
tokenized_X_test = tokenize(X_test)

# 상위 샘플 2개 출력
for sent in tokenized_X_train[:2]:
    print(sent)

## 단어 집합(Vocabulary) 만들기

토큰화된 **훈련 데이터**로부터 정수 인코딩을 위한 단어 집합을 만든다. 검증·테스트 데이터는 단어 집합을 만드는 데 사용하지 않는다는 점에 주의하자 — 모델이 미리 봐서는 안 되는 데이터이기 때문이다.

`collections.Counter` 를 쓰면 데이터에 존재하는 단어 종류의 총 개수와 각 단어의 등장 빈도를 셀 수 있다.

In [ ]:
from collections import Counter

word_list = []
for sent in tokenized_X_train:
    for word in sent:
        word_list.append(word)

word_counts = Counter(word_list)
print('총 단어수 :', len(word_counts))

In [ ]:
print('훈련 데이터에서의 단어 the의 등장 횟수 :', word_counts['the'])
print('훈련 데이터에서의 단어 great의 등장 횟수 :', word_counts['great'])

단어를 **등장 빈도수가 높은 순서대로 정렬**하여 `vocab` 에 저장하고, 빈도수 상위 10개 단어를 출력한다.

In [ ]:
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
print('등장 빈도수 상위 10개 단어')
print(vocab[:10])

## 희귀 단어 제거

빈도수가 낮은 단어들은 자연어 처리에서 배제하고자 한다. 등장 빈도가 `threshold` 미만인 단어들이 데이터에서 얼마만큼의 비중을 차지하는지 확인한다.

> **threshold 노트** : 교안은 훈련 리뷰 2만 개 기준으로 `threshold = 3`(2회 이하 제거)을 사용했고, 그 결과 희귀 단어가 단어 집합의 61%를 차지하지만 전체 등장 빈도에서는 1.3%에 불과함을 확인했다. 이 학습노트의 코퍼스는 훨씬 작으므로 `threshold = 2`(1회 등장 단어 제거)로 조정한다. 핵심 논리 — *"단어 종류로는 많지만 실제 등장 비중은 미미하므로 제거해도 된다"* — 는 동일하다.

In [ ]:
threshold = 2
total_cnt = len(word_counts)   # 단어의 수
rare_cnt = 0    # 등장 빈도수가 threshold보다 작은 단어의 개수를 카운트
total_freq = 0  # 훈련 데이터의 전체 단어 빈도수 총합
rare_freq = 0   # 등장 빈도수가 threshold보다 작은 단어의 등장 빈도수 총합

# 단어와 빈도수의 쌍(pair)을 key와 value로 받는다.
for key, value in word_counts.items():
    total_freq = total_freq + value
    # 단어의 등장 빈도수가 threshold보다 작으면
    if value < threshold:
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기 :', total_cnt)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s' % (threshold - 1, rare_cnt))
print('단어 집합에서 희귀 단어의 비율:', (rare_cnt / total_cnt) * 100)
print('전체 등장 빈도에서 희귀 단어 등장 빈도 비율:', (rare_freq / total_freq) * 100)

In [ ]:
# 전체 단어 개수 중 빈도수가 threshold 미만인 단어는 제거
vocab_size = total_cnt - rare_cnt
vocab = vocab[:vocab_size]
print('단어 집합의 크기 :', len(vocab))

## 스페셜 토큰 : `<PAD>` 와 `<UNK>`

아직 각 단어에 고유한 정수를 부여하지는 않았다. 그 전에 정수 0과 1에는 **특별한 용도의 단어**를 배정한다.

- 정수 **0** : 패딩에 사용하는 패딩 토큰 **`<PAD>`**
- 정수 **1** : OOV(Out-Of-Vocabulary) 문제 발생 시, 즉 단어 집합에 없는 단어에 부여하는 **`<UNK>`**

이렇게 특별한 용도로 사용되는 단어들을 **스페셜 토큰(Special Token)**이라고 한다. 나머지 단어들은 빈도순으로 정수 2부터 차례대로 부여받는다.

In [ ]:
word_to_index = {}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

for index, word in enumerate(vocab):
    word_to_index[word] = index + 2

vocab_size = len(word_to_index)
print('패딩 토큰과 UNK 토큰을 고려한 단어 집합의 크기 :', vocab_size)

## 정수 인코딩

최종 단어 집합 `word_to_index` 를 이용해 각 단어를 정수로 변환하는 `texts_to_sequences()` 를 구현한다. `word_to_index` 에 존재하지 않는 단어가 등장하면 `<UNK>` 에 해당하는 정수 1을 부여한다.

In [ ]:
def texts_to_sequences(tokenized_X_data, word_to_index):
    encoded_X_data = []
    for sent in tokenized_X_data:
        index_sequences = []
        for word in sent:
            try:
                index_sequences.append(word_to_index[word])
            except KeyError:
                index_sequences.append(word_to_index['<UNK>'])   # OOV → 정수 1
        encoded_X_data.append(index_sequences)
    return encoded_X_data

encoded_X_train = texts_to_sequences(tokenized_X_train, word_to_index)
encoded_X_valid = texts_to_sequences(tokenized_X_valid, word_to_index)
encoded_X_test = texts_to_sequences(tokenized_X_test, word_to_index)

# 상위 샘플 2개 출력
for sent in encoded_X_train[:2]:
    print(sent)

정수 인코딩 결과를 역으로 복원(디코딩)해 보기 위해, `word_to_index` 의 key와 value를 뒤집은 `index_to_word` 를 만들고 첫 번째 샘플을 복원한다. 훈련 데이터에서 1회만 등장해 단어 집합에서 제외된 단어가 있었다면 그 자리는 `<UNK>` 로 복원된다.

In [ ]:
index_to_word = {}
for key, value in word_to_index.items():
    index_to_word[value] = key

decoded_sample = [index_to_word[word] for word in encoded_X_train[0]]
print('기존의 첫번째 샘플 :', tokenized_X_train[0])
print('복원된 첫번째 샘플 :', decoded_sample)

## 패딩(Padding)

서로 다른 길이의 데이터들을 **동일한 길이로 일치**시켜 주는 패딩 작업을 진행한다. 이를 위해 먼저 훈련 데이터의 최대 길이, 평균 길이, 길이 분포를 확인한다.

In [ ]:
import matplotlib.pyplot as plt

print('리뷰의 최대 길이 :', max(len(review) for review in encoded_X_train))
print('리뷰의 평균 길이 :', sum(map(len, encoded_X_train)) / len(encoded_X_train))
plt.hist([len(review) for review in encoded_X_train], bins=10)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

모델이 처리할 수 있도록 모든 샘플의 길이를 특정 길이 `max_len` 으로 통일해야 한다. 대부분의 리뷰가 잘리지 않는 최적의 `max_len` 은 얼마일까? 전체 샘플 중 길이가 `max_len` 이하인 샘플의 비율을 확인하는 함수를 만들어 판단한다. (교안의 IMDB에서는 최대 길이 2,818 / 평균 279에서 `max_len = 500` 을 선택해 약 88%의 샘플을 보존했다.)

In [ ]:
def below_threshold_len(max_len, nested_list):
    count = 0
    for sentence in nested_list:
        if len(sentence) <= max_len:
            count = count + 1
    print('전체 샘플 중 길이가 %s 이하인 샘플의 비율: %s' % (max_len, (count / len(nested_list)) * 100))

max_len = 12
below_threshold_len(max_len, encoded_X_train)

패딩 함수 `pad_sequences()` 를 구현한다. 최대 길이보다 **긴** 데이터는 뒷부분을 잘라내고, **짧은** 데이터는 뒤에 0(`<PAD>`)을 채워 길이를 맞춘다. 결과적으로 모든 데이터의 길이가 `max_len` 으로 통일된다.

In [ ]:
def pad_sequences(sentences, max_len):
    features = np.zeros((len(sentences), max_len), dtype=int)
    for index, sentence in enumerate(sentences):
        if len(sentence) != 0:
            features[index, :len(sentence)] = np.array(sentence)[:max_len]
    return features

padded_X_train = pad_sequences(encoded_X_train, max_len=max_len)
padded_X_valid = pad_sequences(encoded_X_valid, max_len=max_len)
padded_X_test = pad_sequences(encoded_X_test, max_len=max_len)

print('훈련 데이터의 크기 :', padded_X_train.shape)
print('검증 데이터의 크기 :', padded_X_valid.shape)
print('테스트 데이터의 크기 :', padded_X_test.shape)

In [ ]:
print('첫번째 샘플의 길이 :', len(padded_X_train[0]))
print('첫번째 샘플 :', padded_X_train[0])

# 3. 배치 구성 : 텐서 변환과 데이터로더

전처리가 끝났으니 넘파이 배열을 파이토치 텐서로 변환하고, 배치 단위 연산을 위한 데이터로더를 구성한다. 먼저 현재 실습 환경에서 GPU를 사용할 수 있는지 확인한다. (Colab에서 GPU 런타임을 선택했다면 `cuda` 가 출력된다.)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")
print("cpu와 cuda 중 다음 기기로 학습함:", device)

레이블 데이터를 파이토치 텐서 타입으로 변환하고 훈련 데이터의 상위 5개 레이블을 출력한다.

In [ ]:
train_label_tensor = torch.tensor(np.array(y_train))
valid_label_tensor = torch.tensor(np.array(y_valid))
test_label_tensor = torch.tensor(np.array(y_test))
print(train_label_tensor[:5])

훈련·검증·테스트 데이터 각각을 `TensorDataset` 으로 묶고 `DataLoader` 로 감싼다.

> **배치 크기 노트** : 교안은 훈련 배치 32(2만 개 ÷ 32 = 총 625 배치), 검증·테스트 배치 1을 사용한다. 이 노트의 데이터는 작으므로 훈련 배치를 8로 줄인다. 검증·테스트를 배치 1로 두는 것은 교안 설정을 그대로 따른 것으로, 평가에서는 기울기 계산이 없어 속도 부담이 적고 어떤 배치 크기든 결과가 같기 때문에 크게 해도 무방하다.

In [ ]:
encoded_train = torch.tensor(padded_X_train).to(torch.int64)
train_dataset = torch.utils.data.TensorDataset(encoded_train, train_label_tensor)
train_dataloader = torch.utils.data.DataLoader(train_dataset, shuffle=True, batch_size=8)

encoded_valid = torch.tensor(padded_X_valid).to(torch.int64)
valid_dataset = torch.utils.data.TensorDataset(encoded_valid, valid_label_tensor)
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, shuffle=True, batch_size=1)

encoded_test = torch.tensor(padded_X_test).to(torch.int64)
test_dataset = torch.utils.data.TensorDataset(encoded_test, test_label_tensor)
test_dataloader = torch.utils.data.DataLoader(test_dataset, shuffle=True, batch_size=1)

total_batch = len(train_dataloader)
print('총 배치의 수 : {}'.format(total_batch))

# 4. RNN 계열 모델로 분류하기 : LSTM과 GRU (13-02 · 13-03)

모델을 구현할 때는 **각 층을 지날 때마다 텐서의 크기가 어떻게 변하는지** 이해하는 것이 중요하다. 교안의 설정(단어 벡터 차원 100, 배치 크기 32, 문장 길이 500, 은닉 상태 차원 128, 카테고리 2개)을 기준으로 모델 내부의 데이터 변화를 정리하면 다음과 같다.

```text
(32, 500)                 입력 데이터 (배치 크기, 문장 길이)
   → 임베딩 층 통과 후 →  (32, 500, 100)   (배치 크기, 문장 길이, 임베딩 벡터의 차원)
   → LSTM/GRU 통과 후 →   (32, 128)        (배치 크기, 은닉 상태의 차원)  ← 마지막 시점만
   → 출력층 통과 후 →     (32, 2)          (배치 크기, 분류할 카테고리의 수)
```

텍스트 분류는 다-대-일 문제이므로 RNN 계열 층의 **마지막 시점의 은닉 상태(hidden state)** 하나만 출력층 `nn.Linear` 에 연결한다. 은닉 상태가 모든 시점(문장 길이)만큼 존재하는 것이 아니라 단 하나만 전달된다는 점이 핵심이다. 출력층은 소프트맥스 회귀를 수행하므로 (배치 크기, 카테고리 수)의 차원을 가진다.

## LSTM 분류기 (13-02)

`nn.LSTM` 은 전체 시점의 출력 `lstm_out` 과 함께 **(hidden state, cell state)의 튜플**을 반환한다. 이 중 마지막 시점의 은닉 상태 `hidden` 만 사용한다. `hidden` 의 크기는 (1, 배치 크기, hidden_dim)이므로 `squeeze(0)` 으로 첫 번째 차원을 제거해 (배치 크기, hidden_dim)으로 만든 뒤 출력층에 넣는다.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)   # output_dim = 분류하고자 하는 카테고리의 개수

    def forward(self, x):
        # x: (batch_size, seq_length)
        embedded = self.embedding(x)   # (batch_size, seq_length, embedding_dim)
        # LSTM은 (hidden state, cell state)의 튜플을 반환한다.
        # lstm_out: (batch_size, seq_length, hidden_dim), hidden: (1, batch_size, hidden_dim)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        last_hidden = hidden.squeeze(0)   # (batch_size, hidden_dim)
        logits = self.fc(last_hidden)     # (batch_size, output_dim)
        return logits

## GRU 분류기 (13-03)

GRU 버전도 구조는 완전히 동일하다. 유일한 차이는 반환값이다.

> **LSTM vs GRU 반환값** : `nn.LSTM` 은 `output, (hidden, cell)` 을 반환하지만, `nn.GRU` 는 셀 상태가 없으므로 `output, hidden` 을 반환한다. 튜플 언패킹 부분만 다르고 나머지 코드는 같다.

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch_size, seq_length)
        embedded = self.embedding(x)   # (batch_size, seq_length, embedding_dim)
        # GRU는 셀 상태 없이 hidden만 반환한다.
        # gru_out: (batch_size, seq_length, hidden_dim), hidden: (1, batch_size, hidden_dim)
        gru_out, hidden = self.gru(embedded)
        last_hidden = hidden.squeeze(0)   # (batch_size, hidden_dim)
        logits = self.fc(last_hidden)     # (batch_size, output_dim)
        return logits

## 하이퍼파라미터와 모델 선언

임베딩 벡터의 차원, 은닉 상태의 차원, 학습률처럼 **사용자가 정해 주는 값이면서 모델의 결과에 영향을 미치는 값**들을 **하이퍼파라미터(hyperparameter)**라고 한다. 먼저 LSTM 모델부터 선언한다.

소프트맥스 회귀로 분류를 진행하므로 손실 함수는 `nn.CrossEntropyLoss()` 를 사용한다. 파이토치로 자연어 처리를 하면 가장 많이 쓰게 되는 손실 함수다. 옵티마이저는 Adam, 학습률은 0.001로 정한다.

> **여기서 잠깐!** `nn.CrossEntropyLoss()` 는 비용 함수에 **소프트맥스 함수까지 포함**하고 있다. 따라서 모델의 `forward` 마지막에 소프트맥스를 또 적용하면 안 된다 — 모델은 로짓(logits)을 그대로 반환하고, 확률 변환과 손실 계산은 손실 함수에 맡긴다.

In [ ]:
embedding_dim = 100
hidden_dim = 128
output_dim = 2
num_epochs = 50   # 교안은 대용량 데이터라 5 에포크면 충분하지만, 소형 코퍼스는 더 많은 반복이 필요하다.

torch.manual_seed(2)   # 재현성을 위한 시드 고정
model = LSTMClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 5. 평가 코드 작성과 학습 (13-02 ~ 13-04 공통)

## 정확도 함수

모델의 정확도를 측정하는 `calculate_accuracy()` 를 작성한다. 로짓에서 가장 큰 값의 인덱스(`argmax`)가 예측 클래스이고, 이를 실제 레이블과 비교해 맞힌 비율을 계산한다.

In [ ]:
def calculate_accuracy(logits, labels):
    # _, predicted = torch.max(logits, 1) 과 동일한 역할
    predicted = torch.argmax(logits, dim=1)
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    accuracy = correct / total
    return accuracy

## 평가 함수 : `model.eval()` 과 `torch.no_grad()`

검증·테스트 데이터에 대한 성능을 측정하는 `evaluate()` 를 작성한다. 이 함수에 등장하는 두 구문은 모델 평가에서 중요한 역할을 하므로 확실히 짚고 넘어간다.

- **`model.eval()`** : 모델을 **평가 모드**로 설정한다. 드롭아웃이나 배치 정규화처럼 학습과 평가 시 다르게 동작하는 층이 있기 때문에 이 설정이 중요하다. 평가 모드를 설정하지 않으면 이런 층들이 학습 때처럼 동작해 **평가 결과 자체가 틀어질 수 있다.**
- **`with torch.no_grad():`** : 자동 미분 엔진의 **기울기(gradient) 계산을 비활성화**한다. 평가 중에는 기울기가 필요 없으므로 메모리를 절약하고 속도를 높일 수 있다. 적용하지 않아도 평가 결과 자체에는 영향이 없다.

> **정리** : `model.eval()` 은 평가 시 **반드시** 사용해야 하고(결과 정확성), `torch.no_grad()` 는 필수는 아니지만 메모리·속도 측면에서 **권장**된다(효율).

In [ ]:
def evaluate(model, valid_dataloader, criterion, device):
    val_loss = 0
    val_correct = 0
    val_total = 0
    model.eval()
    with torch.no_grad():
        # 데이터로더로부터 배치 크기만큼의 데이터를 연속으로 로드
        for batch_X, batch_y in valid_dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            # 모델의 예측값
            logits = model(batch_X)

            # 손실을 계산
            loss = criterion(logits, batch_y)

            # 정확도와 손실을 누적
            val_loss += loss.item()
            val_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
            val_total += batch_y.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= len(valid_dataloader)
    return val_loss, val_accuracy

## 학습 루프

이제 LSTM 모델을 학습한다. 학습은 `num_epochs` 만큼 반복되며 한 에포크의 흐름은 다음과 같다.

1. `train_dataloader` 에서 배치 단위로 데이터를 가져와 `device` 에 올린다.
2. 모델이 입력을 처리해 예측값(`logits`)을 출력하고, 실제 정답(`batch_y`)과 비교해 손실(`loss`)을 계산한다.
3. `optimizer.zero_grad()` 로 이전 기울기를 초기화하고, `loss.backward()` 로 역전파하여 기울기를 구한 뒤, `optimizer.step()` 으로 가중치를 갱신한다.
4. 에포크가 끝나면 훈련 손실/정확도와 검증 손실/정확도를 계산해 성능을 모니터링한다.
5. 검증 손실(`val_loss`)이 이전 최소값(`best_val_loss`)보다 작아지면 해당 시점의 가중치를 **체크포인트(checkpoint)**로 저장한다. 이렇게 하면 학습이 끝났을 때 **검증 성능이 가장 좋았던 모델**을 확보할 수 있다.

(출력이 길어지지 않도록 로그는 5 에포크마다 출력한다. 교안은 총 5 에포크를 돌며 매 에포크 출력한다.)

In [ ]:
best_val_loss = float('inf')   # 검증 손실의 최저값 추적용. 초기값은 매우 큰 값.

for epoch in range(num_epochs):
    # Training
    train_loss = 0
    train_correct = 0
    train_total = 0
    model.train()   # 모델을 훈련 모드로 설정

    for batch_X, batch_y in train_dataloader:
        # Forward pass
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        # batch_X.shape == (batch_size, max_len)
        logits = model(batch_X)

        # Compute loss
        loss = criterion(logits, batch_y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate training accuracy and loss
        train_loss += loss.item()
        train_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
        train_total += batch_y.size(0)

    train_accuracy = train_correct / train_total
    train_loss /= len(train_dataloader)

    # Validation
    val_loss, val_accuracy = evaluate(model, valid_dataloader, criterion, device)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')
        print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

    # 검증 손실이 최소일 때 체크포인트 저장
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_lstm.pth')

## 모델 로드 및 평가

학습 과정에서 검증 손실이 최소일 때의 체크포인트를 저장해 두었다. 이를 **베스트 모델**로 판단하고 로드하여 검증·테스트 데이터에 대한 성능을 평가한다.

In [ ]:
# 베스트 모델 로드
model.load_state_dict(torch.load('best_model_lstm.pth'))
model.to(device)

# 검증 데이터에 대한 정확도와 손실 계산
val_loss, val_accuracy = evaluate(model, valid_dataloader, criterion, device)
print(f'Best model validation loss: {val_loss:.4f}')
print(f'Best model validation accuracy: {val_accuracy:.4f}')

In [ ]:
# 테스트 데이터에 대한 정확도와 손실 계산
test_loss, test_accuracy = evaluate(model, test_dataloader, criterion, device)
print(f'Best model test loss: {test_loss:.4f}')
print(f'Best model test accuracy: {test_accuracy:.4f}')

## 모델 테스트 : `predict()` 함수

전처리되지 않은 임의의 리뷰 텍스트를 받아 예측하는 `predict()` 함수를 작성한다. 학습 때와 **완전히 동일한 전처리**(토큰화 → 소문자화 → 정수 인코딩)를 거쳐야 한다는 점이 핵심이다. 단어 집합에 없는 단어(OOV)는 `<UNK>` 토큰의 인덱스 1을 할당한다.

In [ ]:
index_to_tag = {0: '부정', 1: '긍정'}

def predict(text, model, word_to_index, index_to_tag):
    # 모델 평가 모드
    model.eval()

    # 토큰화 및 정수 인코딩. OOV 문제 발생 시 <UNK> 토큰에 해당하는 인덱스 1 할당
    tokens = simple_tokenize(text)   # 교안: word_tokenize(text) 후 token.lower()
    token_indices = [word_to_index.get(token, 1) for token in tokens]

    # 1D CNN 모델도 함께 쓸 수 있도록, 커널 크기(5)보다 짧은 입력은 <PAD>로 채운다.
    if len(token_indices) < 5:
        token_indices = token_indices + [0] * (5 - len(token_indices))

    # 리스트를 텐서로 변경
    input_tensor = torch.tensor([token_indices], dtype=torch.long).to(device)   # (1, seq_length)

    # 모델의 예측
    with torch.no_grad():
        logits = model(input_tensor)   # (1, output_dim)

    # 레이블 인덱스 예측
    _, predicted_index = torch.max(logits, dim=1)   # (1,)

    # 인덱스와 매칭되는 카테고리 문자열로 변경
    predicted_tag = index_to_tag[predicted_index.item()]
    return predicted_tag

부정적인 리뷰와 긍정적인 리뷰를 각각 넣어 모델이 잘 예측하는지 테스트한다.

In [ ]:
test_input = "what a boring waste of time the acting was terrible and the story was dumb"
print(predict(test_input, model, word_to_index, index_to_tag))

In [ ]:
test_input = "the film was a beautiful masterpiece with brilliant acting and I loved the touching story"
print(predict(test_input, model, word_to_index, index_to_tag))

## 학습 루프의 함수화와 GRU 학습

위 학습 루프는 이후 GRU, CNN에서도 그대로 반복되므로 함수 `train_model()` 로 정리해 재사용한다. 내용물은 위의 명시적 루프와 완전히 동일하다.

In [ ]:
def train_model(model, train_dataloader, valid_dataloader, num_epochs, checkpoint_path, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        train_loss = 0
        train_correct = 0
        train_total = 0
        model.train()
        for batch_X, batch_y in train_dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
            train_total += batch_y.size(0)

        train_accuracy = train_correct / train_total
        train_loss /= len(train_dataloader)
        val_loss, val_accuracy = evaluate(model, valid_dataloader, criterion, device)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}: '
                  f'Train Loss {train_loss:.4f}, Train Acc {train_accuracy:.4f} | '
                  f'Val Loss {val_loss:.4f}, Val Acc {val_accuracy:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)

    # 베스트 체크포인트 로드 후 반환
    model.load_state_dict(torch.load(checkpoint_path))
    return model

In [ ]:
torch.manual_seed(2)
gru_model = GRUClassifier(vocab_size, embedding_dim, hidden_dim, output_dim).to(device)
gru_model = train_model(gru_model, train_dataloader, valid_dataloader,
                        num_epochs=50, checkpoint_path='best_model_gru.pth')

val_loss, val_accuracy = evaluate(gru_model, valid_dataloader, criterion, device)
test_loss, test_accuracy = evaluate(gru_model, test_dataloader, criterion, device)
print(f'GRU validation accuracy: {val_accuracy:.4f}')
print(f'GRU test accuracy: {test_accuracy:.4f}')

> **교안 결과 비교** : 교안 기준 네이버 리뷰 + LSTM은 테스트 정확도 **84.92%**, IMDB + GRU는 **86.41%**를 기록했다. 이 노트의 소형 코퍼스는 데이터가 적어 실행마다 수치 변동이 크므로, 절대 수치보다는 파이프라인과 학습 곡선(검증 손실 기준 체크포인트)이 어떻게 동작하는지에 집중하자.

# 6. 1D CNN으로 텍스트 분류하기 (13-04)

합성곱 신경망을 자연어 처리에 사용하기 위한 **1D CNN**을 이해하고, 텍스트 분류에 적용한다.

## 2D 합성곱 복습

이미지 처리에서의 합성곱 연산은 다음과 같이 정의했었다.

> **합성곱 연산** : 커널(kernel) 또는 필터(filter)라는 $n \times m$ 크기의 행렬로 높이 × 너비 크기의 이미지를 처음부터 끝까지 겹치며 훑으면서, 겹쳐지는 부분의 이미지와 커널의 원소 값을 곱해서 모두 더한 값을 출력으로 하는 것. 이미지의 가장 왼쪽 위부터 오른쪽 아래까지 순차적으로 훑는다.

이처럼 커널이 **높이와 너비 두 방향**으로 모두 이동하는 것을 **2D 합성곱**이라 부른다.

## 1D 합성곱 : 문장 행렬 위의 합성곱

LSTM 실습을 상기해 보면, 각 문장은 임베딩 층을 지나 **각 단어가 임베딩 벡터가 된 문장 행렬** 상태로 입력되었다. 1D 합성곱의 입력도 이와 동일하다. 문장의 길이를 $n$, 임베딩 벡터의 차원을 $k$ 라 하면 문장은 $n \times k$ 행렬이 된다.

핵심적인 차이는 **커널의 너비가 임베딩 벡터의 차원 $k$ 와 동일하게 고정**된다는 점이다. 커널이 너비 방향으로는 더 이상 움직일 곳이 없으므로, **문장 행렬의 높이 방향(단어 방향)으로만 이동**한다. 그래서 1D 합성곱에서는 커널의 높이만으로 커널의 크기를 말한다. 예를 들어 'wait for the video and do n't rent it'이라는 문장(길이 9)에 크기 2인 커널을 적용하면, 첫 스텝에서 'wait for', 둘째 스텝에서 'for the', 셋째 스텝에서 'the video' … 순서로 두 단어씩 훑어 내려간다.

한 번의 연산을 1 **스텝(step)**이라 할 때, 크기 $K$ 의 커널이 길이 $n$ 의 문장을 전부 훑으면 결과 벡터의 길이는 다음과 같다.

$$ L_{out} = n - K + 1 $$

길이 9 문장에 크기 2 커널이면 8차원, 크기 3 커널이면 7차원 벡터를 얻는다.

## 커널 크기의 의미 : n-gram

CNN의 커널은 신경망 관점에서 **가중치 행렬**이므로 커널 크기에 따라 학습되는 파라미터 수가 달라진다. 자연어 처리 관점에서는 커널 크기에 따라 **한 번에 참고하는 단어 묶음의 크기**가 달라진다 — 즉, 참고하는 **n-gram**이 달라진다.

| 커널의 크기 | 각 스텝에서 참고하는 것 |
|:---:|:---:|
| 2 | bigram |
| 3 | trigram |

## 맥스 풀링(Max-pooling)

이미지 CNN과 마찬가지로 합성곱 층(합성곱 연산 + 활성화 함수) 다음에는 풀링 층을 추가한다. 대표적인 것이 **맥스 풀링**으로, 합성곱 결과 벡터에서 **가장 큰 값 하나(스칼라)**를 빼내는 연산이다.

교안의 설계 예시는 크기 4 커널 2개, 크기 3 커널 2개, 크기 2 커널 2개, 총 6개의 커널을 사용한다. 길이 9 문장에서 각각 6차원 × 2, 7차원 × 2, 8차원 × 2의 결과 벡터를 얻고, 맥스 풀링으로 6개의 스칼라를 뽑은 뒤 전부 **연결(concatenate)**해 하나의 벡터로 만든다. 이 벡터가 곧 **1D CNN이 문장으로부터 얻은 문장 벡터**이며, 이를 뉴런 2개짜리 출력층에 완전 연결(`nn.Linear()`)하여 분류를 수행한다.

## `nn.Conv1d` 의 입출력 크기 실험

파이토치의 `nn.Conv1d` 는 입력으로 (배치 크기, **임베딩 벡터의 차원**, 문장 길이)를 기대한다. 임베딩 층의 출력은 (배치 크기, 문장 길이, 임베딩 벡터의 차원)이므로, 뒤에서 `permute` 로 축을 바꿔 주게 된다. 크기 변화를 먼저 실험해 본다.

In [ ]:
# input.shape == (배치 크기, 임베딩 벡터의 차원, 문장 길이)
input = torch.randn(32, 16, 50)

# 선언 시 nn.Conv1d(임베딩 벡터의 차원, 커널의 개수, 커널 사이즈)
m = nn.Conv1d(16, 33, 3, stride=1)

# output.shape == (배치 크기, 커널의 개수, 컨볼루션 연산 결과 벡터)
output = m(input)
print(output.shape)   # 연산 결과 벡터의 길이 = 50 - 3 + 1 = 48

## CNN 모델 클래스

교안의 CNN은 단순화를 위해 **크기 5짜리 커널 한 종류만 256개** 사용한다. `forward` 에서 텐서 크기의 변화를 주석으로 따라가 보자. 특히 맥스 풀링을 별도의 풀링 층 없이 `max(1)[0]` 으로 구현한 부분이 눈여겨볼 포인트다 — 컨볼루션 결과 축을 두 번째 차원으로 옮긴 뒤(`permute`) 그 축에 대해 최댓값을 취하면, 커널(필터) 256개 각각에 대해 스칼라 하나씩 총 256차원의 문장 벡터가 남는다.

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self, vocab_size, num_labels):
        super(CNN, self).__init__()
        # 오직 하나의 종류의 필터만 사용함
        self.num_filter_sizes = 1   # 윈도우 5짜리 1개만 사용
        self.num_filters = 256
        self.word_embed = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=128, padding_idx=0)

        # 윈도우 5짜리 1개만 사용
        self.conv1 = torch.nn.Conv1d(128, self.num_filters, 5, stride=1)
        self.dropout = torch.nn.Dropout(0.5)
        self.fc1 = torch.nn.Linear(1 * self.num_filters, num_labels, bias=True)

    def forward(self, inputs):
        # word_embed(inputs).shape == (배치 크기, 문장 길이, 임베딩 벡터의 차원)
        # word_embed(inputs).permute(0, 2, 1).shape == (배치 크기, 임베딩 벡터의 차원, 문장 길이)
        embedded = self.word_embed(inputs).permute(0, 2, 1)

        # max를 이용한 maxpooling
        # conv1(embedded).shape == (배치 크기, 커널 개수, 컨볼루션 연산 결과)
        # conv1(embedded).permute(0, 2, 1).shape == (배치 크기, 컨볼루션 연산 결과, 커널 개수)
        # conv1(embedded).permute(0, 2, 1).max(1)[0].shape == (배치 크기, 커널 개수)
        x = F.relu(self.conv1(embedded).permute(0, 2, 1).max(1)[0])

        # y_pred.shape == (배치 크기, 분류할 카테고리의 수)
        y_pred = self.fc1(self.dropout(x))
        return y_pred

임베딩 층에 `padding_idx=0` 을 지정한 점도 확인하자. `<PAD>` 토큰(정수 0)의 임베딩 벡터를 0벡터로 고정하고 기울기 갱신에서도 제외한다.

같은 데이터로더와 `train_model()` 을 그대로 재사용해 학습하고 평가한다.

In [ ]:
torch.manual_seed(2)
cnn_model = CNN(vocab_size, num_labels=len(set(y_train))).to(device)
cnn_model = train_model(cnn_model, train_dataloader, valid_dataloader,
                        num_epochs=50, checkpoint_path='best_model_cnn.pth')

val_loss, val_accuracy = evaluate(cnn_model, valid_dataloader, criterion, device)
test_loss, test_accuracy = evaluate(cnn_model, test_dataloader, criterion, device)
print(f'CNN validation accuracy: {val_accuracy:.4f}')
print(f'CNN test accuracy: {test_accuracy:.4f}')

`predict()` 함수도 그대로 사용할 수 있다.

> **주의 — CNN과 입력 최소 길이** : 커널 크기가 5이므로 CNN에는 **길이 5 미만의 입력을 넣을 수 없다**(컨볼루션 결과 길이가 $L - 5 + 1 \le 0$). 앞서 `predict()` 안에 짧은 입력을 `<PAD>` 로 채우는 방어 코드를 넣어 둔 이유다. 교안의 IMDB 리뷰는 충분히 길어 이 문제가 드러나지 않는다.

In [ ]:
test_input = "what a boring waste of time the acting was terrible and the story was dumb"
print(predict(test_input, cnn_model, word_to_index, index_to_tag))

In [ ]:
test_input = "the film was a beautiful masterpiece with brilliant acting and I loved the touching story"
print(predict(test_input, cnn_model, word_to_index, index_to_tag))

# 7. 사전 훈련된 임베딩으로 성능 올리기 (13-05)

지금까지의 모델은 `nn.Embedding` 을 **무작위 초기화** 상태에서 시작해 분류 데이터만으로 임베딩을 학습했다. 데이터가 충분히 크지 않으면 단어의 의미를 임베딩이 제대로 담기 어렵다. 이때 12챕터(Day 8)에서 배운 **사전 훈련된 워드 임베딩(pre-trained word embedding)** — 대규모 코퍼스에서 미리 학습된 Word2Vec, GloVe 등 — 을 임베딩 층의 초기값으로 주입하면 성능을 끌어올릴 수 있다.

교안은 구글이 학습해 둔 **GoogleNews Word2Vec(300차원)**을 gensim으로 로드한다.

> **원본 실습 코드 노트** : 아래 코드는 약 1.5GB의 파일을 내려받으므로 이 노트에서는 주석으로 남긴다. Colab에서는 그대로 실행하면 된다.
>
> ```python
> # !pip install gensim gdown
> # !gdown https://drive.google.com/uc?id=1Av37IVBQAAntSe1X3MOAl5gvowQzd2_j
> # import gensim
> # word2vec_model = gensim.models.KeyedVectors.load_word2vec_format(
> #     'GoogleNews-vectors-negative300.bin.gz', binary=True)
> ```

이 노트에서는 매핑 과정을 어디서나 실행해 볼 수 있도록, 감성 단어들이 긍정/부정 두 군집으로 미리 학습되어 있는 상황을 흉내 낸 **데모용 '사전 훈련' 벡터**를 만든다. 긍정 단어들은 공통 기준 벡터 근처에, 부정 단어들은 그 반대 방향 근처에 배치한다 — 실제 Word2Vec에서 의미가 비슷한 단어들이 가까이 모이는 성질의 축소판이다.

In [ ]:
rng = np.random.default_rng(42)

# 데모용 '사전 훈련' 임베딩이 커버하는 단어들 (실제 Word2Vec처럼 일부 단어는 없을 수 있다)
pretrained_pos_words = ['great', 'amazing', 'brilliant', 'masterpiece', 'loved', 'beautiful',
                        'wonderful', 'touching', 'superb', 'joy', 'best', 'outstanding',
                        'recommended', 'enjoyed', 'smiling', 'unforgettable', 'moving', 'heart']
pretrained_neg_words = ['terrible', 'awful', 'boring', 'dumb', 'waste', 'hated', 'disappointing',
                        'worst', 'horrible', 'dull', 'lazy', 'bad', 'pointless', 'annoying',
                        'predictable', 'cheap', 'weak', 'poor', 'fake', 'angry', 'regretted',
                        'mess', 'boredom', 'painful', 'forgettable', 'disaster']

base_pos = rng.normal(0, 1, 300)   # 긍정 단어들의 기준 벡터
base_neg = -base_pos               # 부정 단어들은 반대 방향

word2vec_model = {}
for w in pretrained_pos_words:
    word2vec_model[w] = (base_pos + rng.normal(0, 0.1, 300)).astype(np.float32)
for w in pretrained_neg_words:
    word2vec_model[w] = (base_neg + rng.normal(0, 0.1, 300)).astype(np.float32)

print('데모 사전 훈련 임베딩이 커버하는 단어 수 :', len(word2vec_model))

## 임베딩 행렬(embedding_matrix) 만들기

사전 훈련된 임베딩은 300차원이므로, **우리 단어 집합의 크기만큼의 행**과 **300차원의 열**을 가지는 행렬을 만든다. 그리고 우리 토크나이저 기준의 정수 인덱스에 맞춰 사전 훈련 벡터를 맵핑한다. 예를 들어 우리 단어 집합에서 'great'가 13번이라면, `embedding_matrix` 의 13번 행에 사전 훈련 임베딩의 'great' 벡터를 넣는 식이다.

`<PAD>`(0번)와 `<UNK>`(1번)는 실제 단어가 아니므로 맵핑에서 제외하며, 사전 훈련 임베딩에 없는 단어의 행도 0벡터로 남는다.

> **여기서 잠깐 — 교안 코드의 인덱스 조건** : 교안은 "0번과 1번은 맵핑에서 제외"라고 설명하면서 코드는 `if i > 2:` 를 사용한다. 이 조건이면 **정수 2번 단어까지 제외**되어 주석과 어긋난다(off-by-one). 의도대로라면 `if i > 1:` 이 맞으므로 여기서는 수정해서 작성한다.

In [ ]:
embedding_matrix = np.zeros((vocab_size, 300))
print('임베딩 행렬의 크기 :', embedding_matrix.shape)

def get_vector(word):
    if word in word2vec_model:
        return word2vec_model[word]
    else:
        return None

# <PAD>를 위한 0번과 <UNK>를 위한 1번은 실제 단어가 아니므로 맵핑에서 제외
for word, i in word_to_index.items():
    if i > 1:
        temp = get_vector(word)
        if temp is not None:
            embedding_matrix[i] = temp

`<PAD>` 나 `<UNK>` 에는 사전 훈련된 임베딩이 들어가지 않았으므로 0번 행은 0벡터여야 한다. 확인해 보자.

In [ ]:
# <PAD>나 <UNK>의 경우는 사전 훈련된 임베딩이 들어가지 않아서 0벡터임
embedding_matrix[0]

이제 특정 단어의 벡터가 **정확하게 맵핑되었는지** 검증한다. 현재 단어 집합에서 'great'가 몇 번 정수에 맵핑되어 있는지 확인하고, 사전 훈련 임베딩에서의 'great' 벡터와 임베딩 행렬의 해당 행이 일치하는지 `np.all` 로 비교한다.

In [ ]:
print("'great'의 정수 인덱스 :", word_to_index['great'])

# 사전 훈련 임베딩에서의 'great' 벡터와 embedding_matrix의 해당 행이 일치하는지 체크
np.all(word2vec_model['great'] == embedding_matrix[word_to_index['great']])

## 사전 훈련된 임베딩을 사용하는 CNN

앞의 CNN과 동일하되, 워드 임베딩 층의 가중치에 `embedding_matrix` 를 **`nn.Parameter` 로 주입**하는 부분이 핵심이다. `embedding_dim` 도 사전 훈련 벡터의 차원인 300으로 바뀌고, `conv1` 의 입력 채널 수도 그에 맞춰 300이 된다.

`requires_grad = True` 로 두면 사전 훈련 벡터를 초기값으로 삼아 분류 학습 중에 **함께 미세 조정(fine-tuning)**한다.

> **동결(freeze)하고 싶다면?** `requires_grad = False` 로 바꾸면 임베딩이 고정된다. 또는 `nn.Embedding.from_pretrained(tensor, freeze=True)` 를 써도 같은 효과를 얻는다. 데이터가 아주 작을 때는 동결이 과적합을 막는 데 유리할 수 있다.

In [ ]:
class CNNPretrained(torch.nn.Module):
    def __init__(self, vocab_size, num_labels):
        super(CNNPretrained, self).__init__()
        # 오직 하나의 종류의 필터만 사용함
        self.num_filter_sizes = 1   # 윈도우 5짜리 1개만 사용
        self.num_filters = 256

        # 주석 처리된 코드는 기존의 임베딩 층을 사용할 경우
        # self.word_embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=128, padding_idx=0)
        self.word_embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=300)
        self.word_embed.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.word_embed.weight.requires_grad = True   # 사전 훈련 벡터를 초기값으로 파인튜닝

        # 윈도우 5짜리 1개만 사용
        self.conv1 = torch.nn.Conv1d(300, self.num_filters, 5, stride=1)
        self.dropout = torch.nn.Dropout(0.5)
        self.fc1 = torch.nn.Linear(1 * self.num_filters, num_labels, bias=True)

    def forward(self, inputs):
        # word_embed(inputs).shape == (배치 크기, 문장 길이, 임베딩 벡터의 차원)
        embedded = self.word_embed(inputs).permute(0, 2, 1)
        # max를 이용한 maxpooling
        x = F.relu(self.conv1(embedded).permute(0, 2, 1).max(1)[0])
        y_pred = self.fc1(self.dropout(x))
        return y_pred

In [ ]:
torch.manual_seed(2)
cnn_pre_model = CNNPretrained(vocab_size, num_labels=len(set(y_train))).to(device)
cnn_pre_model = train_model(cnn_pre_model, train_dataloader, valid_dataloader,
                            num_epochs=50, checkpoint_path='best_model_cnn_pretrained.pth')

val_loss, val_accuracy = evaluate(cnn_pre_model, valid_dataloader, criterion, device)
test_loss, test_accuracy = evaluate(cnn_pre_model, test_dataloader, criterion, device)
print(f'CNN + 사전 훈련 임베딩 validation accuracy: {val_accuracy:.4f}')
print(f'CNN + 사전 훈련 임베딩 test accuracy: {test_accuracy:.4f}')

In [ ]:
test_input = "what a boring waste of time the acting was terrible and the story was dumb"
print(predict(test_input, cnn_pre_model, word_to_index, index_to_tag))

In [ ]:
test_input = "the film was a beautiful masterpiece with brilliant acting and I loved the touching story"
print(predict(test_input, cnn_pre_model, word_to_index, index_to_tag))

## 결과 해석

무작위 초기화 임베딩의 CNN은 소형 코퍼스에서 단어 의미를 스스로 학습해야 하므로 성능이 흔들리는 반면, 사전 훈련(을 흉내 낸) 임베딩을 주입한 CNN은 감성 단어들이 이미 두 군집으로 구분되어 있어 훨씬 안정적으로 높은 성능을 낸다. **데이터가 작을수록 사전 훈련 임베딩의 효과가 크다**는 것이 이 실습의 요점이다.

교안의 실제 IMDB 실험 결과도 같은 방향을 보여 준다.

| 모델 (교안) | 검증 정확도 | 테스트 정확도 |
|:---|:---:|:---:|
| GRU (13-03) | 86.78% | 86.41% |
| 1D CNN (13-04) | 88.16% | 87.28% |
| 1D CNN + 사전 훈련 임베딩 (13-05) | **89.96%** | **89.83%** |

# 8. 마무리 정리

## 텍스트 분류 파이프라인 요약

| 단계 | 핵심 도구/함수 | 비고 |
|:---|:---|:---|
| 데이터 점검 | `df.info()`, `.isnull().values.any()`, `value_counts()` | 결측값·레이블 균형 확인 |
| 분할 | `train_test_split(..., stratify=y)` | 레이블 비율 유지 |
| 정제 | 정규 표현식 `re.sub` | 한글 `[^ㄱ-ㅎㅏ-ㅣ가-힣 ]` |
| 토큰화 | 영어: `word_tokenize` / 한국어: 형태소 분석기(Mecab) | 훈련 데이터로만 Vocab 구축 |
| 단어 집합 | `Counter` → 빈도순 정렬 → 희귀 단어 제거 | `<PAD>`=0, `<UNK>`=1 |
| 정수 인코딩 | `texts_to_sequences()` | OOV → 1 |
| 패딩 | `pad_sequences()` | 길이 분포 보고 `max_len` 결정 |
| 배치 구성 | `TensorDataset` + `DataLoader` | 정수 텐서는 `int64` |

## 모델 3종 비교

| 항목 | LSTM / GRU | 1D CNN |
|:---|:---|:---|
| 문장을 읽는 방식 | 단어를 시점 순서대로 순차 처리 | 커널(n-gram 윈도우)로 병렬적으로 훑음 |
| 문장 벡터 | 마지막 시점의 은닉 상태 `hidden.squeeze(0)` | 컨볼루션 결과의 맥스 풀링 값 (커널 수만큼) |
| 핵심 크기 변화 | (B, L) → (B, L, E) → (B, H) → (B, 2) | (B, L) → (B, L, E) → permute → (B, F) → (B, 2) |
| 주의점 | LSTM은 `(hidden, cell)` 튜플 반환 | 입력 길이 ≥ 커널 크기 필요 |

## 핵심 체크리스트

- `nn.CrossEntropyLoss()` 는 **소프트맥스를 포함**한다. 모델은 로짓을 반환하고, 소프트맥스를 중복 적용하지 않는다.
- 평가 시 **`model.eval()` 은 필수**(드롭아웃·배치 정규화 동작 전환), `torch.no_grad()` 는 효율을 위한 권장 사항.
- 체크포인트는 **검증 손실 최소 시점**에 저장하고, 최종 평가는 베스트 체크포인트를 로드해 수행한다.
- `predict()` 는 학습 때와 **동일한 전처리 경로**를 밟아야 하며, OOV는 `<UNK>`(1)로 처리한다.
- 사전 훈련 임베딩 주입 : `nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))` — `requires_grad` 로 파인튜닝/동결을 선택한다.

다음 챕터(14장)에서는 문장의 각 단어마다 레이블을 예측하는 **시퀀스 레이블링(Sequence Labeling)**으로 넘어간다.